In [5]:
import joblib
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, Lipinski

saved = joblib.load("esol_rf_7features.joblib")

def predict_logS(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("SMILES 无法解析，请检查输入")

    values = {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HDonors": Lipinski.NumHDonors(mol),
        "HAcceptors": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": Lipinski.RingCount(mol)
    }

    features = pd.DataFrame([values])[saved["feature_columns"]]
    return saved["model"].predict(features)[0]

print("愈创木酚预测 logS：", round(predict_logS("COc1ccccc1O"), 3))

愈创木酚预测 logS： -1.346


In [6]:
molecules = {
    "苯甲醚": "COc1ccccc1",
    "愈创木酚": "COc1ccccc1O",
    "邻苯二酚": "Oc1ccccc1O"
}

results = pd.DataFrame([
    {
        "分子": name,
        "SMILES": smiles,
        "预测 logS": round(predict_logS(smiles), 3)
    }
    for name, smiles in molecules.items()
])

display(results.sort_values("预测 logS", ascending=False))

,分子,SMILES,预测 logS
2,邻苯二酚,Oc1ccccc1O,0.271
0,苯甲醚,COc1ccccc1,-1.094
1,愈创木酚,COc1ccccc1O,-1.346


In [7]:
url = (
    "https://raw.githubusercontent.com/deepchem/deepchem/"
    "master/datasets/delaney-processed.csv"
)
reference = pd.read_csv(url)

target = "measured log solubility in mols per litre"
reference["canonical_smiles"] = reference["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s))
)

for name, smiles in molecules.items():
    canonical = Chem.MolToSmiles(Chem.MolFromSmiles(smiles))
    measured = reference.loc[
        reference["canonical_smiles"] == canonical,
        ["Compound ID", target]
    ]

    print(f"{name}：预测 logS = {predict_logS(smiles):.3f}")
    display(measured)

苯甲醚：预测 logS = -1.094


,Compound ID,measured log solubility in mols per litre
693,Anisole,-1.85


愈创木酚：预测 logS = -1.346


,Compound ID,measured log solubility in mols per litre
680,Guaiacol,-1.96
1069,o-Methoxyphenol,-1.96


邻苯二酚：预测 logS = 0.271


,Compound ID,measured log solubility in mols per litre
249,"1,2-Benzenediol",0.62


### 预测示例的解释

苯甲醚和邻苯二酚属于模型训练集；愈创木酚的两条重复记录在建模前均被排除。因此，这三个分子用于展示 SMILES 输入和模型预测流程，不用于计算模型的独立测试性能。模型性能以 01_explore_ESOL.ipynb 中 186 条留出测试记录的结果为准。